## RAG for duolingo

## crear vectores de la bd vectorial

#### Obtener el texto desde los pdfs

In [23]:
import os
import boto3
import pymupdf
import io
from dotenv import load_dotenv

load_dotenv()

def extraer_texto_de_pdf_s3(bucket_name: str, file_key: str) -> str:
    s3_client = boto3.client('s3')
    
    print(f"Descargando '{file_key}' desde el bucket S3 '{bucket_name}'...")
    
    try:
        response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        pdf_bytes = response['Body'].read()
        texto_completo = ""
        with pymupdf.open(stream=pdf_bytes, filetype="pdf") as doc:
            for page in doc:
                texto_completo += page.get_text() + "\n"
                
        print("¡Texto extraído con éxito desde S3!")
        return texto_completo.strip()
        
    except Exception as e:
        print(f"Error al procesar el archivo desde S3: {e}")
        return ""

#### Divir el texto en chunks

In [24]:
import re

def dividir_pdf_duolingo_por_secciones(texto_completo: str) -> list:
    """
    Divide el texto del PDF de Duolingo respetando las secciones numeradas del documento.
    """
    # Expresión regular para detectar los encabezados de sección (ej. "1. Tus primeros pasos", "2. Lecciones...")
    patron_secciones = re.compile(r'\n(?=\d+\.\s+[A-ZÁÉÍÓÚa-záéíóú\s]+)')
    
    # Dividir el texto usando las secciones detectadas
    secciones = patron_secciones.split(texto_completo)
    chunks_estructurados = []
    
    for i, seccion in enumerate(secciones):
        seccion_limpia = seccion.strip()
        if not seccion_limpia:
            continue
            
        # Opcional: Si una sección es muy larga, puedes subdividirla en sub-chunks, 
        # pero para este documento, cada sección principal cabe perfectamente en un chunk de tamaño ideal para embeddings.
        
        # Extraer un título provisional de la primera línea
        lineas = seccion_limpia.split('\n')
        titulo_seccion = lineas[0] if lineas else f"Seccion_{i}"
        
        chunks_estructurados.append({
            "chunk_id": i,
            "section_title": titulo_seccion,
            "text": seccion_limpia,
            "source": "Primeros pasos: cómo aprender idiomas en Duolingo (Cindy Blanco, Ph.D., 2025)",
            "token_estimate": len(seccion_limpia.split()) # Estimación rápida de tamaño
        })
        
    print(f"Total de chunks semánticos generados: {len(chunks_estructurados)}")
    return chunks_estructurados


#### Convertir los chunks en embeddings

In [28]:
import json
import boto3

def generar_embeddings_chunks_aws(chunks_optimizados: list, region_name: str = "us-east-1") -> list:
    """
    Toma los chunks optimizados y les genera su vector de embedding 
    utilizando AWS Bedrock (Amazon Titan Text Embeddings v1).
    
    Args:
        chunks_optimizados (list): Lista de diccionarios con los chunks.
        region_name (str): Región de AWS donde tienes habilitado Bedrock (ej. us-east-1).
        
    Returns:
        list: Lista de chunks enriquecida con la clave 'embedding' (1536 dimensiones).
    """
    # Inicializar el cliente de Bedrock Runtime
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=region_name
    )
    
    # ID del modelo de embeddings de Amazon Titan
    model_id = "amazon.titan-embed-text-v1"
    
    print(f"Generando vectores con AWS Bedrock para {len(chunks_optimizados)} chunks...")
    
    for i, chunk in enumerate(chunks_optimizados):
        # Estructura del payload que exige el modelo Titan de Bedrock
        payload = {
            "inputText": chunk["text"]
        }
        
        try:
            response = bedrock_runtime.invoke_model(
                modelId=model_id,
                contentType="application/json",
                accept="application/json",
                body=json.dumps(payload)
            )
            
            # Leer la respuesta de la API de Bedrock
            response_body = json.loads(response["body"].read())
            
            # Extraer el vector de 1536 dimensiones
            chunk["embedding"] = response_body.get("embedding", [])
            print(f"[{i+1}/{len(chunks_optimizados)}] Vectorizado con Bedrock: {chunk.get('section_title', 'Chunk')}")
            
        except Exception as e:
            print(f"Error al vectorizar el chunk {i} con Bedrock: {e}")
            chunk["embedding"] = []
            
    return chunks_optimizados


#### Insertar los vectores en la BD vectorial

In [40]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

def insertar_chunks_con_vectores_en_rds(chunks_con_vectores: list):
    if not chunks_con_vectores:
        print("La lista de chunks está vacía. No hay nada que insertar.")
        return

    try:
        print("Conectando a Amazon RDS PostgreSQL...")
        connection = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            database="vectorial-rag",
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            port=os.getenv("DB_PORT", "5432")
        )
        cursor = connection.cursor()
        
        # Diagnóstico para verificar la BD activa
        cursor.execute("SELECT current_database();")
        db_actual = cursor.fetchone()[0]
        print(f"🔗 Conectado a la base de datos: '{db_actual}'")
        
        print(f"Insertando {len(chunks_con_vectores)} chunks vectorizados en la base de datos...")
        
        # Usamos public.duolingo_chunks explícitamente para evitar errores de ruta
        sql_insert = """
            INSERT INTO public.duolingo_chunks (section_title, source, chunk_text, embedding)
            VALUES (%s, %s, %s, %s);
        """
        
        for i, chunk in enumerate(chunks_con_vectores):
            section_title = chunk.get("section_title", "Sin título")
            source = chunk.get("source", "Duolingo Thrive Document")
            chunk_text = chunk.get("text", "")
            embedding = chunk.get("embedding", [])
            
            if not embedding:
                continue
                
            cursor.execute(sql_insert, (section_title, source, chunk_text, embedding))
            
        connection.commit()
        print("¡Todos los chunks con vectores fueron guardados con éxito en RDS!")
        
    except Exception as e:
        print(f"❌ Error al insertar los datos en RDS: {e}")
        if 'connection' in locals() and connection:
            connection.rollback()
            
    finally:
        if 'cursor' in locals() and cursor:
            cursor.close()
        if 'connection' in locals() and connection:
            connection.close()
            print("Conexión a RDS cerrada correctamente.")

In [41]:
# 1. Extraer texto y generar chunks desde S3
BUCKET_NAME = "duolingo-thrive-234"
FILE_KEY = "example.pdf"

texto_pdf = extraer_texto_de_pdf_s3(bucket_name=BUCKET_NAME, file_key=FILE_KEY)
chunks_optimizados = dividir_pdf_duolingo_por_secciones(texto_pdf)

# 2. Generar los embeddings utilizando AWS Bedrock (en lugar de Azure)
chunks_con_vectores = generar_embeddings_chunks_aws(chunks_optimizados, region_name="us-east-1")

# 3. Guardar todo en tu base de datos vectorial de AWS RDS
insertar_chunks_con_vectores_en_rds(chunks_con_vectores)

Descargando 'example.pdf' desde el bucket S3 'duolingo-thrive-234'...
¡Texto extraído con éxito desde S3!
Total de chunks semánticos generados: 8
Generando vectores con AWS Bedrock para 8 chunks...
[1/8] Vectorizado con Bedrock: Primeros pasos: cómo aprender idiomas en
[2/8] Vectorizado con Bedrock: 1. Tus primeros pasos en Duolingo
[3/8] Vectorizado con Bedrock: 2. Lecciones y funcionalidades para aprender
[4/8] Vectorizado con Bedrock: 3. Gamificación y motivación
[5/8] Vectorizado con Bedrock: 4. Suscripciones
[6/8] Vectorizado con Bedrock: 5. Ajustes y personalización
[7/8] Vectorizado con Bedrock: 6. Tips para usar Duolingo
[8/8] Vectorizado con Bedrock: 7. Idea central
Conectando a Amazon RDS PostgreSQL...
🔗 Conectado a la base de datos: 'vectorial-rag'
Insertando 8 chunks vectorizados en la base de datos...
¡Todos los chunks con vectores fueron guardados con éxito en RDS!
Conexión a RDS cerrada correctamente.


## Usuario manda mensaje y se procesa

#### El mensaje (string) se convierte en un embedding de las mismas dimenciones

In [42]:
import json
import boto3

def convertir_texto_a_embedding(texto: str, region_name: str = "us-east-1") -> list:
    """
    Toma un texto de entrada, lo convierte en un vector de embedding 
    utilizando AWS Bedrock (Amazon Titan Text Embeddings v1) y lo devuelve.
    
    Args:
        texto (str): El string o pregunta que se quiere vectorizar.
        region_name (str): La región de AWS donde tienes configurado Bedrock.
        
    Returns:
        list: Una lista de floats con el embedding (1536 dimensiones).
    """
    if not texto or not texto.strip():
        print("El texto proporcionado está vacío.")
        return []

    # Inicializar el cliente de Bedrock Runtime
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=region_name
    )
    
    model_id = "amazon.titan-embed-text-v1"
    
    payload = {
        "inputText": texto
    }
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(payload)
        )
        
        # Leer la respuesta y extraer el vector
        response_body = json.loads(response["body"].read())
        embedding = response_body.get("embedding", [])
        
        return embedding
        
    except Exception as e:
        print(f"❌ Error al generar el embedding del texto con Bedrock: {e}")
        return []

#### Busar los k vectores mas cercanos en la BD vectorial con balltree o IVF

In [46]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

def buscar_chunks_cercanos(vector_consulta: list, k: int = 3) -> list:
    """
    Toma un embedding de consulta y busca en la base de datos RDS PostgreSQL 
    los k chunks más cercanos utilizando el índice vectorial (IVF).
    
    Args:
        vector_consulta (list): El embedding de 1536 dimensiones (la pregunta vectorizada).
        k (int): Número de vecinos más cercanos a recuperar (por defecto 3).
        
    Returns:
        list: Una lista de diccionarios con la información de los chunks más similares.
    """
    if not vector_consulta:
        print("El vector de consulta está vacío.")
        return []

    resultados_cercanos = []
    connection = None
    cursor = None

    try:
        # Conexión forzando la base de datos 'vectorial-rag'
        connection = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            database="vectorial-rag",
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            port=os.getenv("DB_PORT", "5432")
        )
        cursor = connection.cursor()

        # Opcional: Aumentar el número de listas a sondear (nprobes) para mejorar la precisión del IVF
        # cursor.execute("SET ivfflat.probes = 2;")

        # Consulta SQL utilizando el operador de distancia de coseno de pgvector (<=>)
        # Nota: Convertimos la lista de Python a formato string vector de PostgreSQL '[val1, val2, ...]'
        vector_str = "[" + ",".join(map(str, vector_consulta)) + "]"

        sql_query = """
            SELECT section_title, source, chunk_text, (embedding <=> %s::vector) AS distancia
            FROM public.duolingo_chunks
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
        """

        # Pasamos el vector dos veces (para el SELECT y para el ORDER BY) y el límite k
        cursor.execute(sql_query, (vector_str, vector_str, k))
        filas = cursor.fetchall()

        for fila in filas:
            section_title, source, chunk_text, distancia = fila
            resultados_cercanos.append({
                "section_title": section_title,
                "source": source,
                "chunk_text": chunk_text,
                "distancia": float(distancia) # Menor distancia = mayor similitud
            })

        print(f"¡Búsqueda completada! Se recuperaron los {len(resultados_cercanos)} chunks más cercanos.")
        return resultados_cercanos

    except Exception as e:
        print(f"❌ Error al realizar la búsqueda vectorial en RDS: {e}")
        return []

    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()

#### Tomar los k vectores, converitrlos a string y pasarlos a LLM como contexto para generar una respuesta

In [51]:
import boto3
from botocore.exceptions import ClientError

def generar_respuesta_rag(pregunta: str, chunks_cercanos: list, model_id: str = "google.gemma-3-12b-it", region_name: str = "us-east-1") -> str:
    """
    Toma la pregunta del usuario y los chunks recuperados de RDS, construye un contexto 
    y consulta a un modelo LLM en AWS Bedrock usando client.converse().
    """
    if not chunks_cercanos:
        return "Lo siento, no encontré información relevante en los documentos para responder a tu pregunta."

    # 1. Unir los textos de los chunks recuperados para armar el contexto
    contexto_textual = ""
    for i, chunk in enumerate(chunks_cercanos, 1):
        contexto_textual += f"\n--- Fragmento {i} [{chunk.get('section_title', 'Sin título')}] ---\n"
        contexto_textual += chunk.get('chunk_text', '') + "\n"

    # 2. Definir el prompt estructurado para el RAG
    prompt_completo = f"""
SISTEMA: Eres 'Duolingo Thrive', un asistente virtual experto y amigable. 
Tu objetivo es responder a la pregunta del usuario utilizando **única y exclusivamente** el contexto proporcionado a continuación. Si la respuesta no se encuentra en el contexto, di amablemente que no dispones de esa información. No inventes datos.

[CONTEXTO DE DOCUMENTOS]
{contexto_textual}

Pregunta del usuario: {pregunta}

Respuesta:
"""

    # 3. Estructura de mensajes compatible con client.converse()
    messages = [
        {
            "role": "user",
            "content": [{"text": prompt_completo}]
        }
    ]

    client = boto3.client("bedrock-runtime", region_name=region_name)

    try:
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "maxTokens": 800,
                "temperature": 0.3,
                "topP": 0.9
            }
        )
        
        # Extracción segura de la respuesta usando converse
        respuesta_ia = response['output']['message']['content'][0]['text']
        return respuesta_ia
        
    except ClientError as e:
        print(f"❌ Error de cliente en Bedrock LLM: {e}")
        return "Ocurrió un error de validación o permisos al consultar el modelo en Bedrock."
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
        return "Ocurrió un error inesperado al intentar generar la respuesta con la Inteligencia Artificial."

#### Pasar respuesta al fronted

In [52]:
# 1. El usuario escribe una pregunta
pregunta_usuario = "¿Cómo puedo ganar XP más rápido y mantener mi racha?"

print(f"Pregunta: {pregunta_usuario}\n")

# 2. Paso A: Convertir la pregunta a embedding con Titan de Bedrock
vector_pregunta = convertir_texto_a_embedding(pregunta_usuario)

# 3. Paso B: Buscar los k=3 fragmentos más cercanos en RDS usando el índice IVF
chunks_relevantes = buscar_chunks_cercanos(vector_pregunta, k=3)

# 4. Paso C: Enviar la pregunta y los fragmentos recuperados al LLM de Bedrock para la respuesta final
respuesta_final = generar_respuesta_rag(pregunta_usuario, chunks_relevantes)

print("\n=== RESPUESTA DEL CHATBOT ===")
print(respuesta_final)

Pregunta: ¿Cómo puedo ganar XP más rápido y mantener mi racha?

¡Búsqueda completada! Se recuperaron los 3 chunks más cercanos.

=== RESPUESTA DEL CHATBOT ===
Para ganar EXP más rápido, puedes completar lecciones, prácticas, Cuentos y desafíos contrarreloj. Las Ligas te permiten competir semanalmente y avanzar de división según la EXP obtenida. Para mantener tu racha, puedes usar Protectores de racha si olvidas practicar.
